# ProAID — B2.2: CEFR-Calibrated Detector (3 Thresholds)
**PT-L2Detect Benchmark** | Google Colab + GPU T4

Train AI detector với features: XLM-R CLS + linguistic + CEFR-gap.
**Per-group calibration**: τ_Beg, τ_Int, τ_Adv learned via ROC-optimized threshold on validation set.

Compare với B2.1 (single threshold) để verify Claims C2 & C3.

**⚠️ IMPORTANT — CEFR-gap leakage prevention**:
CEFR-gap expected stats MUST be computed from **HUMAN-ONLY** essays in train set.
Using all essays (human+AI) causes circular leakage: expected stats "know" the label.
In deployment, reference distributions come from human learner corpora, not AI outputs.

**Prerequisite**: B1 model + data đã có trên Google Drive (`MyDrive/proaid/`)

## 0. Setup & Mount Drive

In [ ]:
!pip install -q transformers scikit-learn torch matplotlib seaborn

In [ ]:
import json, os, re, numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
    accuracy_score, roc_auc_score, roc_curve)
from collections import Counter
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive

print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB)')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/proaid'
DATA_DIR = f'{DRIVE_BASE}/data'
MODEL_DIR = f'{DRIVE_BASE}/models'
LOG_DIR = f'{DRIVE_BASE}/logs'

print(f'Data:  {os.path.exists(DATA_DIR)} — {len(os.listdir(DATA_DIR)) if os.path.exists(DATA_DIR) else 0} files')
print(f'Models: {os.path.exists(MODEL_DIR)} — {os.listdir(MODEL_DIR) if os.path.exists(MODEL_DIR) else "(empty)"}')

# Quick check prerequisites
for f in ['data/train_full.jsonl', 'models/best_model.pt']:
    path = f'{DRIVE_BASE}/{f}'
    assert os.path.exists(path), f'❌ Missing: {path}. Run B0 + B1 first!'
print('✅ All prerequisites found on Drive')

## 1. Load B1 CEFR Model from Drive

In [ ]:
# Recreate architecture + load weights
CEFR_3CLASS = ["Beginner", "Intermediate", "Advanced"]
CEFR_TO_3CLASS = {"A1":"Beginner","A2":"Beginner","B1":"Intermediate","B2":"Intermediate","C1":"Advanced"}

class CEFRClassifier3(nn.Module):
    def __init__(self, model_name, num_labels=3, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden,128),
            nn.ReLU(), nn.Dropout(dropout), nn.Linear(128,num_labels))
    def forward(self, ids, am):
        return self.classifier(self.encoder(input_ids=ids, attention_mask=am).last_hidden_state[:,0,:])

cefr_ckpt = torch.load(f'{MODEL_DIR}/best_model.pt', map_location=DEVICE)
cefr_model = CEFRClassifier3(cefr_ckpt.get('model_name','xlm-roberta-base'), 3).to(DEVICE)
cefr_model.load_state_dict(cefr_ckpt['model_state_dict'])
cefr_model.eval()
cefr_tokenizer = AutoTokenizer.from_pretrained(cefr_ckpt.get('model_name','xlm-roberta-base'))
print(f'✅ B1 CEFR model loaded (WF1: {cefr_ckpt.get("val_wf1","?")})')

## 2. Linguistic Features + CEFR-gap Features

In [ ]:
def extract_linguistic_features(text: str) -> np.ndarray:
    words = text.split(); n_words = len(words)
    f_avg_word_len = np.mean([len(w) for w in words]) if words else 0
    f_ttr = len(set(w.lower() for w in words)) / n_words if n_words else 0
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 0]
    n_sentences = max(len(sentences), 1)
    f_avg_sent_len = n_words / n_sentences
    f_punct_ratio = sum(1 for c in text if c in ',;:()"\'-') / len(text) if text else 0
    f_upper_ratio = sum(1 for c in text if c.isupper()) / len(text) if text else 0
    f_comma_per_sent = text.count(',') / n_sentences
    f_short_word_ratio = sum(1 for w in words if len(w) <= 3) / n_words if n_words else 0
    return np.array([n_words, f_avg_word_len, f_ttr, f_avg_sent_len,
                     f_punct_ratio, f_upper_ratio, f_comma_per_sent, f_short_word_ratio], dtype=np.float32)

feat_names = ['word_count','avg_word_len','ttr','avg_sent_len',
              'punct_ratio','upper_ratio','comma_per_sent','short_word_ratio']
LING_DIM = GAP_DIM = len(feat_names)

## 3. Load Data + Predict CEFR + Compute CEFR-gap

In [ ]:
def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl(f'{DATA_DIR}/train_full.jsonl')
val_data   = load_jsonl(f'{DATA_DIR}/val_full.jsonl')
test_data  = load_jsonl(f'{DATA_DIR}/test_full.jsonl')

# Predict 3-class labels (on ALL data)
def predict_cefr_batch(texts, batch_size=32):
    preds = []
    for i in range(0, len(texts), batch_size):
        bt = texts[i:i+batch_size]
        enc = cefr_tokenizer(bt, truncation=True, padding=True, max_length=512, return_tensors='pt')
        with torch.no_grad():
            logits = cefr_model(enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE))
            preds.extend([CEFR_3CLASS[p] for p in torch.argmax(logits,-1).cpu().numpy()])
    return preds

print('🔮 Predicting 3-class CEFR labels...')
for ds_name, ds in [('train', train_data), ('val', val_data), ('test', test_data)]:
    pred_classes = predict_cefr_batch([e['text'] for e in ds])
    for e, pc in zip(ds, pred_classes):
        e['pred_class'] = pc
    print(f'  {ds_name}: {dict(sorted(Counter(pc for pc in pred_classes).items()))}')

# Extract linguistic features
print('\n🔧 Extracting linguistic features...')
for ds in [train_data, val_data, test_data]:
    for e in ds:
        e['ling_feats'] = extract_linguistic_features(e['text'])

# ============================================================
# ⚠️ CRITICAL FIX: CEFR-gap expected stats from HUMAN-ONLY
# ============================================================
# Using all essays (human+AI) would cause circular leakage:
#   expected_stats already "knows" AI patterns → gap = 0 for AI
#   → model just looks at gap magnitude to detect AI
# In deployment, reference distributions come from HUMAN learner corpora only.
print('\n📐 Computing CEFR-gap from TRAIN **HUMAN-ONLY** expected distributions...')
train_human = [e for e in train_data if not e['is_ai']]
print(f'  Train human essays: {len(train_human)} (vs {len(train_data)} total)')

expected_stats = {}
for grp in CEFR_3CLASS:
    grp_feats = np.array([e['ling_feats'] for e in train_human if e['pred_class'] == grp])
    if len(grp_feats) == 0:
        continue
    expected_stats[grp] = {'mean': grp_feats.mean(0), 'std': grp_feats.std(0) + 1e-8}
    print(f'  {grp}: n={len(grp_feats)} human, mean wc={expected_stats[grp]["mean"][0]:.1f}')

# Compute CEFR-gap = (observed - human_expected) / human_std
for ds in [train_data, val_data, test_data]:
    for e in ds:
        grp = e['pred_class']
        if grp in expected_stats:
            e['gap_feats'] = ((e['ling_feats'] - expected_stats[grp]['mean']) / expected_stats[grp]['std']).astype(np.float32)
        else:
            e['gap_feats'] = np.zeros(LING_DIM, dtype=np.float32)

print(f'\n✅ Features: {LING_DIM} ling + {GAP_DIM} gap = {LING_DIM+GAP_DIM} explicit')
print(f'   CEFR-gap expected stats from HUMAN-ONLY → no circular leakage')

## 4. Dataset & Model (+ CEFR-gap)

In [ ]:
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 512
BATCH_SIZE = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class CalibratedDetectorDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data; self.tokenizer = tokenizer; self.max_length = max_length
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        e = self.data[idx]
        enc = self.tokenizer(e['text'], truncation=True, padding='max_length',
                             max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0), 'attention_mask': enc['attention_mask'].squeeze(0),
                'ling_feats': torch.tensor(e['ling_feats'], dtype=torch.float),
                'gap_feats': torch.tensor(e['gap_feats'], dtype=torch.float),
                'is_ai': torch.tensor(1.0 if e['is_ai'] else 0.0, dtype=torch.float),
                'cefr': e['cefr_level'], 'pred_class': e['pred_class']}

train_ds = CalibratedDetectorDataset(train_data, tokenizer, MAX_LENGTH)
val_ds   = CalibratedDetectorDataset(val_data, tokenizer, MAX_LENGTH)
test_ds  = CalibratedDetectorDataset(test_data, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f'Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)} batches')

In [ ]:
class AIDetectorCalibrated(nn.Module):
    def __init__(self, model_name, ling_dim=8, gap_dim=8, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        total_dim = self.encoder.config.hidden_size + ling_dim + gap_dim
        self.classifier = nn.Sequential(nn.Linear(total_dim,256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128,1))
    def forward(self, ids, am, lf, gf):
        xlmr_cls = self.encoder(input_ids=ids, attention_mask=am).last_hidden_state[:,0,:]
        return self.classifier(torch.cat([xlmr_cls, lf, gf], dim=-1)).squeeze(-1)

model = AIDetectorCalibrated(MODEL_NAME, LING_DIM, GAP_DIM).to(DEVICE)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

## 5. Training

In [ ]:
EPOCHS, LR, WARMUP, WEIGHT_DECAY, GRAD_CLIP, PATIENCE = 5, 2e-5, 0.1, 0.01, 1.0, 2

n_human = sum(1 for e in train_data if not e['is_ai'])
n_ai = sum(1 for e in train_data if e['is_ai'])
pos_weight = torch.tensor([n_human / n_ai]).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps*WARMUP), total_steps)
print(f'pos_weight: {pos_weight.item():.3f}, Steps: {total_steps}')

In [ ]:
def evaluate_calibrated(model, loader, thresholds=None):
    """Evaluate với per-group thresholds. thresholds: {grp: tau} or None → τ=0.5"""
    model.eval()
    loss_total, probs, labels, cefrs, pred_classes = 0, [], [], [], []
    with torch.no_grad():
        for batch in loader:
            ids, am, lf, gf, lb = batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE), batch['ling_feats'].to(DEVICE), batch['gap_feats'].to(DEVICE), batch['is_ai'].to(DEVICE)
            logits = model(ids, am, lf, gf)
            loss_total += criterion(logits, lb).item()
            probs.extend(torch.sigmoid(logits).cpu().numpy())
            labels.extend(lb.cpu().numpy())
            cefrs.extend(batch['cefr']); pred_classes.extend(batch['pred_class'])
    probs, labels = np.array(probs), np.array(labels)
    if thresholds is None:
        thresholds = {g: 0.5 for g in CEFR_3CLASS}
    preds = np.array([1 if probs[i] >= thresholds.get(pc, 0.5) else 0 for i, pc in enumerate(pred_classes)])

    per_cefr, per_group = {}, {}
    for lvl in ['A1','A2','B1','B2','C1']:
        mask = np.array([c == lvl for c in cefrs])
        if mask.sum() == 0: continue
        g_l, g_p, g_pr = labels[mask], preds[mask], probs[mask]
        tn, fp, fn, tp = confusion_matrix(g_l, g_p, labels=[0,1]).ravel()
        per_cefr[lvl] = {'f1': f1_score(g_l,g_p), 'fpr': fp/(fp+tn) if (fp+tn)>0 else 0,
            'acc': accuracy_score(g_l,g_p), 'auc': roc_auc_score(g_l,g_pr) if len(set(g_l))>1 else 0.5, 'n': int(mask.sum())}
    for grp in CEFR_3CLASS:
        mask = np.array([c == grp for c in pred_classes])
        if mask.sum() == 0: continue
        g_l, g_p, g_pr = labels[mask], preds[mask], probs[mask]
        tn, fp, fn, tp = confusion_matrix(g_l, g_p, labels=[0,1]).ravel()
        per_group[grp] = {'f1': f1_score(g_l,g_p), 'fpr': fp/(fp+tn) if (fp+tn)>0 else 0,
            'acc': accuracy_score(g_l,g_p), 'auc': roc_auc_score(g_l,g_pr) if len(set(g_l))>1 else 0.5, 'n': int(mask.sum())}
    return {'loss': loss_total/len(loader), 'f1': f1_score(labels,preds), 'auc': roc_auc_score(labels,probs),
        'acc': accuracy_score(labels,preds), 'probs': probs, 'labels': labels,
        'cefrs': cefrs, 'pred_classes': pred_classes, 'per_cefr': per_cefr, 'per_group': per_group,
        'thresholds': thresholds}

def train_epoch_cal(ep):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        ids, am, lf, gf, lb = batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE), batch['ling_feats'].to(DEVICE), batch['gap_feats'].to(DEVICE), batch['is_ai'].to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(ids, am, lf, gf), lb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()
        if (step+1) % 20 == 0:
            print(f'  Ep {ep} | {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}')
    return total_loss / len(train_loader)

# === RUN ===
history = {'train_loss': [], 'val_f1': [], 'val_auc': []}
best_f1, best_ep, patience_cnt = 0, 0, 0

print(f'{"="*60}')
print(f'B2.2 CEFR-CALIBRATED DETECTOR')
print(f'{"="*60}')

for epoch in range(1, EPOCHS+1):
    tl = train_epoch_cal(epoch)
    vr = evaluate_calibrated(model, val_loader)  # default τ=0.5
    history['train_loss'].append(tl)
    history['val_f1'].append(vr['f1'])
    history['val_auc'].append(vr['auc'])
    print(f'  → Ep {epoch:2d} | TL: {tl:.4f} | Val F1: {vr["f1"]:.4f} | Val AUC: {vr["auc"]:.4f}')
    if vr['f1'] > best_f1:
        best_f1, best_ep, patience_cnt = vr['f1'], epoch, 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_f1': vr['f1'], 'val_auc': vr['auc']},
                   f'{MODEL_DIR}/calibrated_detector.pt')
        print(f'  💾 Saved to Drive (F1={best_f1:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'⏹ Early stop epoch {epoch}'); break

print(f'\n✅ Best Val F1 (τ=0.5): {best_f1:.4f} (epoch {best_ep})')

## 6. Learn Optimal Per-Group Thresholds on VALIDATION

In [ ]:
ckpt = torch.load(f'{MODEL_DIR}/calibrated_detector.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])

val_eval = evaluate_calibrated(model, val_loader)
thresholds_range = np.linspace(0.01, 0.99, 99)
optimal_thresholds = {}

print('🔧 Learning optimal thresholds on VALIDATION set:')
for grp in CEFR_3CLASS:
    mask = np.array([c == grp for c in val_eval['pred_classes']])
    if mask.sum() == 0:
        optimal_thresholds[grp] = 0.5; continue
    f1s = [f1_score(val_eval['labels'][mask], (val_eval['probs'][mask]>=t).astype(int)) for t in thresholds_range]
    best_t = thresholds_range[np.argmax(f1s)]
    optimal_thresholds[grp] = best_t
    print(f'  {grp}: τ = {best_t:.3f} (F1-max = {max(f1s):.4f})')

print(f'\n📊 Threshold comparison:')
for grp in CEFR_3CLASS:
    print(f'  {grp}: global 0.5 → calibrated {optimal_thresholds[grp]:.3f} (Δ={optimal_thresholds[grp]-0.5:+.3f})')

## 7. Test Set: Single τ=0.5 vs Calibrated τ ⭐ KEY RESULT

In [ ]:
tr_single = evaluate_calibrated(model, test_loader, thresholds={g: 0.5 for g in CEFR_3CLASS})
tr_cal = evaluate_calibrated(model, test_loader, thresholds=optimal_thresholds)

print(f'{"="*60}')
print(f'📊 COMPARISON: Single τ=0.5 vs CEFR-Calibrated τ')
print(f'{"="*60}')
print(f'  {"Metric":<15} {"Single τ=0.5":<15} {"Calibrated":<15} {"Δ":<10}')
print(f'  {"-"*55}')
for metric in ['f1','auc','acc']:
    print(f'  {metric:<15} {tr_single[metric]:<15.4f} {tr_cal[metric]:<15.4f} {tr_cal[metric]-tr_single[metric]:+.4f}')

print(f'\n📋 Per-CEFR Breakdown:')
print(f'  {"Level":<6} {"N":<6} {"F1(τ=0.5)":<11} {"F1(cal)":<11} {"FPR(τ=0.5)":<12} {"FPR(cal)":<12} {"ΔF1":<8} {"ΔFPR":<8}')
print(f'  {"-"*74}')
for lvl in ['A1','A2','B1','B2','C1']:
    s = tr_single['per_cefr'].get(lvl, {}); c = tr_cal['per_cefr'].get(lvl, {})
    if not s: continue
    print(f'  {lvl:<6} {s["n"]:<6} {s["f1"]:<11.4f} {c["f1"]:<11.4f} {s["fpr"]:<12.4f} {c["fpr"]:<12.4f} {c["f1"]-s["f1"]:+.4f}   {c["fpr"]-s["fpr"]:+.4f}')

# Claims
df1_pct = (tr_cal['f1'] - tr_single['f1']) / tr_single['f1'] * 100
print(f'\n🎯 Claim C2: ΔF1 ≥ 15%? {df1_pct:.1f}% {"✅" if df1_pct >= 15 else "⚠️"}')

beg_s, beg_c = tr_single['per_group'].get('Beginner',{}), tr_cal['per_group'].get('Beginner',{})
if beg_s and beg_c and beg_s['fpr'] > 0:
    dfpr_pct = (beg_s['fpr'] - beg_c['fpr']) / beg_s['fpr'] * 100
    print(f'🎯 Claim C3: Beginner FPR reduction ≥ 25%? {dfpr_pct:.1f}% {"✅" if dfpr_pct >= 25 else "⚠️"}')
    print(f'   Beginner FPR: {beg_s["fpr"]:.4f} → {beg_c["fpr"]:.4f}')

## 8. Visualization: Before/After Calibration

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
levels = ['A1','A2','B1','B2','C1']
x, w = np.arange(len(levels)), 0.35

f1_s = [tr_single['per_cefr'].get(l,{}).get('f1',0) for l in levels]
f1_c = [tr_cal['per_cefr'].get(l,{}).get('f1',0) for l in levels]
axes[0].bar(x-w/2, f1_s, w, label='Single τ=0.5', color='steelblue')
axes[0].bar(x+w/2, f1_c, w, label='Calibrated', color='darkorange')
axes[0].set_xticks(x); axes[0].set_xticklabels(levels); axes[0].set_ylabel('F1')
axes[0].set_title('F1 by CEFR Level', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3, axis='y')

fpr_s = [tr_single['per_cefr'].get(l,{}).get('fpr',0) for l in levels]
fpr_c = [tr_cal['per_cefr'].get(l,{}).get('fpr',0) for l in levels]
axes[1].bar(x-w/2, fpr_s, w, label='Single τ=0.5', color='steelblue')
axes[1].bar(x+w/2, fpr_c, w, label='Calibrated', color='darkorange')
axes[1].set_xticks(x); axes[1].set_xticklabels(levels); axes[1].set_ylabel('FPR')
axes[1].set_title('FPR by CEFR Level', fontweight='bold'); axes[1].legend(); axes[1].grid(alpha=0.3, axis='y')

groups = CEFR_3CLASS
taus = [optimal_thresholds[g] for g in groups]
axes[2].bar(groups, taus, color=['steelblue','darkorange','crimson'])
axes[2].axhline(y=0.5, color='black', linestyle='--', alpha=0.5, label='Global τ=0.5')
for i, t in enumerate(taus):
    axes[2].text(i, t+0.01, f'{t:.3f}', ha='center', fontweight='bold')
axes[2].set_ylabel('Threshold'); axes[2].set_title('Optimal Per-Group Thresholds', fontweight='bold')
axes[2].legend(); axes[2].set_ylim(0, 1)

plt.tight_layout(); plt.savefig(f'{LOG_DIR}/calibration_comparison.png', dpi=150); plt.show()
print(f'💾 Saved to Drive: {LOG_DIR}/calibration_comparison.png')

## 9. Save Results to Drive

In [ ]:
log = {
    'experiment': 'B2.2 — CEFR-Calibrated Detector',
    'timestamp': datetime.now().isoformat(),
    'model': MODEL_NAME, 'device': str(DEVICE),
    'features': {'xlmr_dim': 768, 'ling_dim': LING_DIM, 'gap_dim': GAP_DIM},
    'cefr_classifier_wf1': cefr_ckpt.get('val_wf1', '?'),
    'train_size': len(train_data), 'val_size': len(val_data), 'test_size': len(test_data),
    'optimal_thresholds': {g: float(t) for g, t in optimal_thresholds.items()},
    'results': {
        'single_tau': {'f1': float(tr_single['f1']), 'auc': float(tr_single['auc']), 'acc': float(tr_single['acc'])},
        'calibrated': {'f1': float(tr_cal['f1']), 'auc': float(tr_cal['auc']), 'acc': float(tr_cal['acc'])},
        'delta_f1_pct': float(df1_pct),
        'per_cefr_single': {l: {k: float(v) if isinstance(v, (np.floating,np.integer)) else v
                                for k,v in tr_single['per_cefr'].get(l,{}).items()} for l in levels},
        'per_cefr_calibrated': {l: {k: float(v) if isinstance(v, (np.floating,np.integer)) else v
                                   for k,v in tr_cal['per_cefr'].get(l,{}).items()} for l in levels},
    },
    'claims': {'C2_dF1_15pct': df1_pct >= 15, 'C3_fpr_reduce_25pct': ('dfpr_pct' in dir() and dfpr_pct >= 25)}
}

with open(f'{LOG_DIR}/experiment_log_b2.2.json', 'w', encoding='utf-8') as f:
    json.dump(log, f, indent=2, ensure_ascii=False, default=str)

print(f'✅ All saved to Drive:')
print(f'   Model: {MODEL_DIR}/calibrated_detector.pt')
print(f'   Log:   {LOG_DIR}/experiment_log_b2.2.json')
print(f'   Plot:  {LOG_DIR}/calibration_comparison.png')